In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import requests
import zipfile
import io
import os

# --- 1. Configuration & Hyperparameters ---
BATCH_SIZE = 64
EPOCHS = 100  # Higher is better, but takes longer
LATENT_DIM = 256  # Latent dimensionality of the encoding space
NUM_SAMPLES = 10000  # Train on first 10,000 samples to save RAM/Time
DATA_PATH = '/content/jpn.txt'

# --- 2. Data Acquisition (Download & Unzip) ---

# --- 3. Data Preprocessing ---
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()

with open(DATA_PATH, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")

# Loop over lines and extract pairs
for line in lines[: min(NUM_SAMPLES, len(lines) - 1)]:
    # Format: "English \t Japanese \t Attribution"
    parts = line.split("\t")
    if len(parts) >= 2:
        input_text = parts[0]
        target_text = parts[1]

        # We use "tab" as the "start sequence" character
        # for the targets, and "\n" as "end sequence" character.
        target_text = "\t" + target_text + "\n"

        input_texts.append(input_text)
        target_texts.append(target_text)

        for char in input_text:
            if char not in input_characters:
                input_characters.add(char)
        for char in target_text:
            if char not in target_characters:
                target_characters.add(char)

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print(f"Number of samples: {len(input_texts)}")
print(f"Number of unique input tokens: {num_encoder_tokens}")
print(f"Number of unique output tokens: {num_decoder_tokens}")

# Create token dictionaries
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

# --- 4. Vectorization (One-Hot Encoding) ---
print("Vectorizing data...")
encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32"
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    encoder_input_data[i, t + 1 :, input_token_index[" "]] = 1.0 # Padding

    for t, char in enumerate(target_text):
        # decoder_input_data is ahead of decoder_target_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0

    decoder_input_data[i, t + 1 :, target_token_index[" "]] = 1.0 # Padding
    decoder_target_data[i, t:, target_token_index[" "]] = 1.0 # Padding

# --- 5. Build Model Architecture ---
# Encoder
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))
encoder = keras.layers.LSTM(LATENT_DIM, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c] # We discard outputs, keep states

# Decoder
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))
# We set up our decoder to return full output sequences,
# and to return internal states as well (needed for inference mode).
decoder_lstm = keras.layers.LSTM(LATENT_DIM, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Define the model that will turn `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"]
)

# --- 6. Train the Model ---
print("Training model...")
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,
)

# --- 7. Inference (Sampling) Models ---
# Restore the model and construct the encoder and decoder.
encoder_inputs = model.input[0]  # input_1
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output  # lstm_1
encoder_states = [state_h_enc, state_c_enc]
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_inputs = model.input[1]  # input_2
decoder_state_input_h = keras.Input(shape=(LATENT_DIM,))
decoder_state_input_c = keras.Input(shape=(LATENT_DIM,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

# Reverse-lookup token index to decode sequences back to something readable.
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())

def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length or find stop character.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]

    return decoded_sentence

# --- 8. Test the Results ---
print("-" * 20)
print("Testing decoded sentences:")
for seq_index in range(10):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index : seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence.strip())

Number of samples: 10000
Number of unique input tokens: 72
Number of unique output tokens: 1436
Vectorizing data...
Training model...
Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step - accuracy: 0.6301 - loss: 2.8522 - val_accuracy: 0.6239 - val_loss: 1.9642
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.6769 - loss: 1.6365 - val_accuracy: 0.6538 - val_loss: 1.8358
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.6978 - loss: 1.5558 - val_accuracy: 0.6891 - val_loss: 1.6941
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.7277 - loss: 1.4455 - val_accuracy: 0.6956 - val_loss: 1.6121
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - accuracy: 0.7360 - loss: 1.3773 - val_accuracy: 0.7006 - val_loss: 1.5707
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.7501 - loss: 1.3050 - val_accuracy: 0.7164 - val_loss: 1.5145
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.7570 - loss: 1.26

In [ ]:
def translate_user_input(input_sentence):
    # 1. Matrix Initialization
    # Create a single batch (1 sequence) filled with zeros
    encoder_input_data = np.zeros(
        (1, max_encoder_seq_length, num_encoder_tokens), dtype="float32"
    )

    # 2. One-Hot Encoding
    # We map the input string into the vector space
    for t, char in enumerate(input_sentence):
        # We must ignore characters that the model has never seen (e.g., specific emojis)
        if char in input_token_index:
            encoder_input_data[0, t, input_token_index[char]] = 1.0

    # 3. Padding
    # The model expects the sequence to be padded with spaces to match the fixed length
    # Start padding from the character after the last input character
    encoder_input_data[0, t + 1 :, input_token_index[" "]] = 1.0

    # 4. Decode
    # Feed the prepared vector into our existing decoding function
    decoded_sentence = decode_sequence(encoder_input_data)

    return decoded_sentence

# --- Interactive Loop ---
print("Translation Model Ready!")
print("Enter an English sentence to translate.")
print("Type 'q' or 'quit' to exit the loop.\n")

while True:
    user_input = input("English: ")

    if user_input.lower() in ['q', 'quit']:
        print("Exiting...")
        break

    # Optional: Basic cleaning to match training data style (lowercase, trim)
    # The training data was likely mixed case, but cleaning helps consistency.
    # user_input = user_input.strip()

    try:
        translation = translate_user_input(user_input)
        print(f"Japanese: {translation.strip()}")
        print("-" * 30)
    except Exception as e:
        print(f"Error: {e}")
        print("Try a simpler sentence using common words.")

Translation Model Ready!
Enter an English sentence to translate.
Type 'q' or 'quit' to exit the loop.

English: hi how are you
Japanese: だってるよ。
------------------------------
English: ok ok 
Japanese: 私んてごみれ！
------------------------------
English: bye
Japanese: わっして！
------------------------------
English: go
Japanese: ほんほー！
------------------------------
English: go
Japanese: ほんほー！
------------------------------
English: q
Exiting...
